# 🔍 Fraud Detection — ClearFlow Financial
### End-to-End ML Pipeline · Canadian Fintech Portfolio Project

**Business context:**  
ClearFlow Financial processes card transactions across Interac, Visa Debit, and 
Mastercard rails. With a ~2.5% fraud rate across 200,000+ monthly transactions, 
the business faces two competing costs: missing fraud (direct financial loss, 
~$220 CAD per incident) and blocking legitimate transactions (customer trust and churn).

**DS objective:**  
Build a binary fraud classifier that maximizes recall while keeping false positive 
rate low enough to avoid disrupting legitimate customers — evaluated on AUC-PR 
(not accuracy) due to severe class imbalance.

**Three fraud types in this dataset:**
- **Account Takeover (ATO)** — stolen credentials, new device, different location
- **Card Testing** — many small transactions testing stolen card details  
- **High-Value One-Shot** — single large fraudulent transaction, often international

**Stack:** Python · LightGBM · SHAP · scikit-learn · imbalanced-learn  
**Data:** Four-table synthetic dataset reflecting realistic fintech complexity

## 0. Importing Libraries

In [2]:
import warnings
warnings.filterwarnings('ignore')

# Data manipulation
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

# Machine learning
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    classification_report, confusion_matrix,
    average_precision_score, roc_auc_score,
    precision_recall_curve, roc_curve, f1_score
)
from imblearn.over_sampling import SMOTE
import lightgbm as lgb
import shap

# Plotting defaults
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor':   'white',
    'axes.spines.top':  False,
    'axes.spines.right':False,
    'font.family':      'DejaVu Sans',
    'axes.titlesize':   13,
    'axes.labelsize':   11,
})

# ClearFlow brand palette
CF_BLUE   = '#1A73E8'
CF_DARK   = '#0D1B2A'
CF_RED    = '#E84B1A'
CF_GREY   = '#F4F6F8'
CF_GREEN  = '#1AE8A0'

print("Libraries loaded ✓")
print(f"pandas {pd.__version__} · numpy {np.__version__} · lightgbm {lgb.__version__}")

Libraries loaded ✓
pandas 3.0.3 · numpy 2.4.6 · lightgbm 4.6.0


## 1. Business Context & Problem Definition

### The Business Problem

ClearFlow Financial operates in a zero-sum environment: every dollar of fraud 
caught is a dollar saved, but every legitimate transaction incorrectly blocked 
costs customer trust — the most valuable asset a challenger fintech has.

**The two costs we are optimizing against:**

| Error Type | Technical Name | Business Impact |
|------------|---------------|-----------------|
| Missed fraud | False Negative | Direct loss ~$220 CAD per incident |
| Blocked legit transaction | False Positive | Customer friction, churn risk, ops cost ~$8 CAD |

**Why this is harder than it looks:**
- Fraud rate is ~2.5% — a model predicting "not fraud" always scores 97.5% accuracy
- The two error types are measured in different units (dollars vs. trust)
- Three distinct fraud patterns require different detection signals
- Fraud patterns shift over time — a model trained in January may degrade by June

### How We Will Measure Success

We will **not** use accuracy. Our primary metrics are:

- **AUC-PR** (Area Under Precision-Recall Curve) — primary metric, reflects 
  performance on the minority class under real-world conditions
- **Fraud Recall** — what % of actual fraud do we catch?
- **False Positive Rate** — what % of legitimate transactions do we block?
- **Estimated CAD savings** — translates model performance into business language

### Industry Note
> In production, ClearFlow's fraud team would set the decision threshold based 
> on a cost matrix — not the default 0.5. We will replicate this exactly: tune 
> the threshold on validation data using real cost assumptions, then report 
> business impact in CAD on the held-out test set.

In [3]:
# ── BUSINESS COST ASSUMPTIONS ─────────────────────────────────────────────────
# In production these numbers come from Finance & Customer Success teams
# Here we use realistic Canadian fintech industry benchmarks as placeholders

COST_FALSE_NEGATIVE = 220.00  # CAD — average loss per missed fraud incident
                               # Source: industry benchmark ($150-$300 range)
                               # In production: derived from chargeback history

COST_FALSE_POSITIVE = 8.00    # CAD — cost per incorrectly blocked transaction
                               # Breakdown:
                               #   ~$5 customer service intervention
                               #   ~$3 estimated churn impact (p(churn) × LTV)
                               # In production: derived from CS team + cohort analysis

# Fraud rate observed in this dataset
OBSERVED_FRAUD_RATE = 0.025   # 2.5%

print("Business cost assumptions defined:")
print(f"  Cost of missing fraud (FN)     : ${COST_FALSE_NEGATIVE:.2f} CAD")
print(f"  Cost of blocking legit (FP)    : ${COST_FALSE_POSITIVE:.2f} CAD")
print(f"  FN is {COST_FALSE_NEGATIVE/COST_FALSE_POSITIVE:.0f}x more costly than FP")
print(f"\n  These constants will feed directly into Section 10 — Business Impact Analysis")

Business cost assumptions defined:
  Cost of missing fraud (FN)     : $220.00 CAD
  Cost of blocking legit (FP)    : $8.00 CAD
  FN is 28x more costly than FP

  These constants will feed directly into Section 10 — Business Impact Analysis


## 2. Data Loading & Quality Assessment

### The Four Tables

| Table | Grain | Key Columns |
|-------|-------|-------------|
| `transactions.csv` | One row per transaction | `transaction_id`, `customer_id`, `merchant_id`, `device_id` |
| `customers.csv` | One row per customer | `customer_id`, demographics, credit profile |
| `merchants.csv` | One row per merchant | `merchant_id`, category, location, risk score |
| `devices.csv` | One row per transaction | `device_id`, `transaction_id`, fingerprint data |

**Joining strategy:** transactions is our unit of analysis. All other tables 
join onto it — enriching each transaction row with who made it, where it was 
made, and what device was used.

In [19]:
# Load raw data
transactions = pd.read_csv('../data/raw/transactions.csv')
customers = pd.read_csv('../data/raw/customers.csv')
merchants = pd.read_csv('../data/raw/merchants.csv')
devices = pd.read_csv('../data/raw/devices.csv')

print("Raw data loaded:")
print(f"  transactions : {transactions.shape[0]:>7,} rows "
      f"x {transactions.shape[1]:>2} columns")
print(f"  customers    : {customers.shape[0]:>7,} rows "
      f"x {customers.shape[1]:>2} columns")
print(f"  merchants    : {merchants.shape[0]:>7,} rows "
      f"x {merchants.shape[1]:>2} columns")
print(f"  devices      : {devices.shape[0]:>7,} rows "
      f"x {devices.shape[1]:>2} columns")

Raw data loaded:
  transactions : 200,000 rows x 14 columns
  customers    :  10,000 rows x 14 columns
  merchants    :   2,000 rows x  9 columns
  devices      : 200,000 rows x  9 columns


In [20]:
# Assess missing values across all four tables
tables = {
    'transactions': transactions,
    'customers': customers,
    'merchants': merchants,
    'devices': devices,
}

for name, table in tables.items():
    missing = table.isnull().sum()
    missing = missing[missing > 0]
    pct = (missing / len(table) * 100).round(2)

    print(f"{'-' * 45}")
    print(f" {name.upper()}")
    print(f"{'-' * 45}")

    if missing.empty:
        print(" No missing values")
    else:
        for col in missing.index:
            print(f" {col:<25} {missing[col]:>6,} ({pct[col]:>3.1f}%)")
    print()

---------------------------------------------
 TRANSACTIONS
---------------------------------------------
 province                  30,358 (15.2%)
 fraud_type                195,000 (97.5%)

---------------------------------------------
 CUSTOMERS
---------------------------------------------
 annual_income                300 (3.0%)
 email_domain                 500 (5.0%)
 address_verified             800 (8.0%)

---------------------------------------------
 MERCHANTS
---------------------------------------------
 province                     113 (5.7%)
 years_in_business            240 (12.0%)

---------------------------------------------
 DEVICES
---------------------------------------------
 device_age_days           30,000 (15.0%)
 vpn_detected              20,000 (10.0%)



Insights/observations:
TRANSACTIONS
province — 15.2% missing. These are international transactions where a Canadian province doesn't apply. This isn't random missingness — it's structurally missing, meaning the absence itself carries information. An international transaction has no province by definition.
fraud_type — 97.5% missing. This is expected — only 2.5% of transactions are fraud, and fraud_type is only populated for fraudulent ones. We'll never use this as a model feature (that would be cheating — it's essentially the answer). It's metadata for our analysis only.
CUSTOMERS
annual_income — 3.0% missing. Customers who didn't disclose income during onboarding. Common in fintech KYC flows where income is optional.
email_domain — 5.0% missing. Customers who signed up without providing an email, or provided one that couldn't be parsed.
address_verified — 8.0% missing. Customers who haven't completed address verification yet — a genuine KYC gap that could itself be a risk signal.
MERCHANTS
province — 5.7% missing. International merchants have no Canadian province. Same structural missingness as transaction province.
years_in_business — 12.0% missing. Merchant didn't disclose or data wasn't available at onboarding. Genuinely unknown — different from the structural cases above.
DEVICES
device_age_days — 15.0% missing. Device age couldn't be determined — common when cookies are cleared, private browsing is used, or it's a POS terminal with no tracking.
vpn_detected — 10.0% missing. VPN detection service returned no result — network timeout or unrecognized IP range.

How we'll handle each:

![alt text](image.png)

In [24]:
# Check for duplicate primary keys in each table
print("Duplicate key check:")
print(
    f" transactions (transaction_id): "
    f"{transactions['transaction_id'].duplicated().sum()}"
)
print(
    f" customers (customer_id): "
    f"{customers['customer_id'].duplicated().sum()}"    
)
print(
    f" merchants (merchant_id): "
    f"{merchants['merchant_id'].duplicated().sum()}"
)
print(
    f" devices (device_id): "
    f"{devices['device_id'].duplicated().sum()}"
)

Duplicate key check:
 transactions (transaction_id): 0
 customers (customer_id): 0
 merchants (merchant_id): 0
 devices (device_id): 0


In [28]:
# Merge all four tables onto transactions
# Transactions is our unit of analysis - everything joins onto it
df = (
    transactions
    .merge(
        customers, 
        on='customer_id', 
        how='left',
        suffixes=('', '_cust')
    )
    .merge(
        merchants, 
        on='merchant_id', 
        how='left',
        suffixes=('', '_mer')
    )
    .merge(
        devices, 
        on='transaction_id', 
        how='left', 
        suffixes=('', '_dev')
    )
)

print(f"Merged shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"Row count preserved: {len(df) == len(transactions)}")

Merged shape: 200,000 rows x 43 columns
Row count preserved: True


In [29]:
# Sanity check
# Confirm fraud rate survived the merge
fraud_rate = df['is_fraud'].mean() * 100 

# Confirm all four column keys are present
keys_present = all([
    'customer_id' in df.columns,
    'merchant_id' in df.columns,
    'device_id' in df.columns,
    'transaction_id' in df.columns,
])

# Confirm province columns were handled correctly
province_cols = [c for c in df.columns if 'province' in c]

print(f"Fraud rate after merge: {fraud_rate:.2f}%")
print(f"All keys present      : {keys_present}")
print(f"Province columns      : {province_cols}")
print(f"\nColumn list:")
for i, col in enumerate(df.columns, 1):
    print(f" {i:>2}. {col}")

Fraud rate after merge: 2.50%
All keys present      : True
Province columns      : ['province', 'province_cust', 'province_mer']

Column list:
  1. transaction_id
  2. customer_id
  3. merchant_id
  4. device_id
  5. transaction_date
  6. hour_of_day
  7. day_of_week
  8. amount
  9. payment_rail
 10. mcc_group
 11. is_international
 12. province
 13. is_fraud
 14. fraud_type
 15. age
 16. province_cust
 17. income_band
 18. annual_income
 19. credit_score
 20. credit_limit
 21. kyc_status
 22. acquisition_channel
 23. email_domain
 24. phone_verified
 25. address_verified
 26. created_at
 27. tenure_days
 28. mcc_group_mer
 29. country
 30. province_mer
 31. is_international_mer
 32. latitude
 33. longitude
 34. merchant_risk_score
 35. years_in_business
 36. device_id_dev
 37. device_type
 38. os
 39. browser
 40. ip_country
 41. is_new_device
 42. device_age_days
 43. vpn_detected
